In [7]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

# ==============================================================================
# CONFIGURANDO AS PASTAS DOS ARQUIVOS (SISTEMA ADAPTÁVEL)
# ==============================================================================
# 1. Descobre a pasta exata de onde o Python está executando este código agora
diretorio_execucao = Path(os.getcwd())

# 2. O sistema vai "subindo" as pastas uma a uma até encontrar a pasta que contém a pasta 'data'
pasta_raiz = diretorio_execucao
while pasta_raiz.name and not (pasta_raiz / "data").exists():
    if pasta_raiz == pasta_raiz.parent:
        break
    pasta_raiz = pasta_raiz.parent

# 3. Caso o sistema não encontre automaticamente, força a subida de 2 níveis pelo histórico da estrutura
if not (pasta_raiz / "data").exists():
    pasta_raiz = diretorio_execucao.parent.parent

# 4. Vincula a localização dos arquivos diretamente à pasta principal encontrada
SINAN_PATH = pasta_raiz / "data/processed/sinan_esqu_unsup.parquet"
MAPBIOMAS_PATH = pasta_raiz / "data/corpos_dagua/STATISTICS_MAPBIOMAS_AGUA_COL4_CITY-STATE-BIOME-SUB_BASINS.xlsx"
POP_MUNI_PATH = pasta_raiz / "data/populacao/populacao_municipio.csv"
SANEAMENTO_PATH = pasta_raiz / "data/saneamento/br_mdr_snis_municipio_agua_esgoto.csv"

# Período de tempo que escolhemos para estudar: 2015 até 2025
"""ANO_INICIO = 2015
ANO_FIM = 2019"""

ANO_INICIO = 2020
ANO_FIM = 2025

print(f"-> Pasta principal do projeto encontrada em: {pasta_raiz}")
print(f"-> Tudo pronto! Vamos analisar os dados entre {ANO_INICIO} e {ANO_FIM}.")

-> Pasta principal do projeto encontrada em: c:\Users\anapd\Projects\Projeto-Caramujo
-> Tudo pronto! Vamos analisar os dados entre 2020 e 2025.


In [8]:
print("1/7 - Carregando dados de saúde (SINAN)...")
df_sinan = pd.read_parquet(SINAN_PATH)

# Ajustando as datas para o formato padrão do computador
df_sinan["DT_NOTIFIC"] = pd.to_datetime(df_sinan["DT_NOTIFIC"])
df_sinan["ano"] = df_sinan["DT_NOTIFIC"].dt.year

# Filtro: Só vale quem teve exame de fezes POSITIVO e dentro dos anos do nosso estudo
casos_filtrados = df_sinan[
    (pd.to_numeric(df_sinan["AN_QUALI"], errors="coerce") == 1) &
    (df_sinan["ano"].between(ANO_INICIO, ANO_FIM))
].copy()

# Padronizando o código das cidades para ficar mais fácil juntar as planilhas depois
casos_filtrados["code_muni"] = casos_filtrados["ID_MUNICIP"].astype(str).str[:6]

# Somando quantos casos positivos cada cidade teve por ano
casos_anuais = casos_filtrados.groupby(["code_muni", "ano"]).size().reset_index(name="casos")
print(f"-> Sucesso! Dados de saúde processados.")

1/7 - Carregando dados de saúde (SINAN)...
-> Sucesso! Dados de saúde processados.


In [9]:
print("2/7 - Carregando dados de satélite (MapBiomas)...")
agua_tipo = pd.read_excel(MAPBIOMAS_PATH, sheet_name="WATER_CITY_TYPE")

# Ajustando o código das cidades e filtrando os anos certos
agua_tipo["code_muni"] = agua_tipo["code"].astype(str).str[:6]
agua_filtrada = agua_tipo[agua_tipo["year"].between(ANO_INICIO, ANO_FIM)].copy()

# Calculando a média de água que cada cidade teve no período (para evitar distorções de secas ou enchentes)
agua_muni = (
    agua_filtrada.groupby(["code_muni", "municipality", "state"])
    .agg(
        natural_ha=("natural_area_ha", "mean"),
        anthropic_ha=("anthropic_area_ha", "mean"),
        mining_ha=("mining_area_ha", "mean"),
        hydro_ha=("hydroelectric_area_ha", "mean")
    )
    .reset_index()
)

# Subtraindo as hidrelétricas e a mineração para sobrar só os pequenos açudes e canais
agua_muni["açudes_canais_ha"] = agua_muni["anthropic_ha"] - (agua_muni["mining_ha"] + agua_muni["hydro_ha"])
agua_muni["açudes_canais_ha"] = agua_muni["açudes_canais_ha"].clip(lower=0) # Remove qualquer número negativo estranho

print(f"-> Dados de água calculados com sucesso.")

2/7 - Carregando dados de satélite (MapBiomas)...
-> Dados de água calculados com sucesso.


In [10]:
print("3/7 - Processando dados de população do IBGE e juntando as planilhas...")
pop_municipios_raw = pd.read_csv(POP_MUNI_PATH, sep=";", skiprows=3, on_bad_lines="skip", encoding="utf-8")
pop_municipios_raw.columns = pop_municipios_raw.columns.astype(str).str.strip().str.replace('"', '', regex=False)

# Identificando as colunas de anos e transformando a tabela de "deitada" para "em pé" (Melt)
anos_colunas = [str(ano) for ano in range(ANO_INICIO, ANO_FIM + 1) if str(ano) in pop_municipios_raw.columns]
if "Nível" in pop_municipios_raw.columns:
    pop_municipios_raw = pop_municipios_raw.drop(columns=["Nível"])

id_vars = [c for c in pop_municipios_raw.columns if c not in anos_colunas]
pop_long = pd.melt(pop_municipios_raw, id_vars=id_vars, value_vars=anos_colunas, var_name="ano", value_name="populacao")

# Limpando os números de população que vieram com pontos ou traços textuais do IBGE
pop_long["code_muni"] = pop_long[id_vars[0]].astype(str).str[:6]
pop_long["ano"] = pd.to_numeric(pop_long["ano"]).astype("Int64")
pop_long["populacao"] = (
    pop_long["populacao"].astype(str)
    .str.replace(".", "", regex=False).str.replace("-", "", regex=False).str.replace("...", "", regex=False)
)
pop_long["populacao"] = pd.to_numeric(pop_long["populacao"], errors="coerce")
pop_brasil = pop_long[["code_muni", "ano", "populacao"]].dropna(subset=["populacao"]).copy()

# JUNTANDO TUDO: População + Casos de Doença + Dados de Água do Satélite
base_final = pop_brasil.merge(casos_anuais, on=["code_muni", "ano"], how="left")
base_final["casos"] = base_final["casos"].fillna(0).astype(int)
base_final = base_final.merge(agua_muni, on="code_muni", how="left").dropna(subset=["municipality"])

print(f"-> Planilhas integradas.")

3/7 - Processando dados de população do IBGE e juntando as planilhas...
-> Planilhas integradas.


In [11]:
print("4/7 - Carregando dados de esgoto e saneamento (SNIS)...")
df_saneamento_raw = pd.read_csv(SANEAMENTO_PATH, sep=",")

df_saneamento = df_saneamento_raw[df_saneamento_raw["ano"].between(ANO_INICIO, ANO_FIM)].copy()
df_saneamento["code_muni"] = df_saneamento["id_municipio"].astype(str).str[:6]
df_saneamento["ano"] = df_saneamento["ano"].astype("Int64")

# Separando apenas a coluna de esgoto coletado
df_esgoto_limpo = df_saneamento[["code_muni", "ano", "indice_coleta_esgoto"]].copy()
df_esgoto_limpo["taxa_esgoto"] = pd.to_numeric(df_esgoto_limpo["indice_coleta_esgoto"], errors="coerce")
df_esgoto_limpo = df_esgoto_limpo.drop(columns=["indice_coleta_esgoto"])

# Unindo o esgoto na nossa base final e calculando a taxa de incidência final
base_final = base_final.merge(df_esgoto_limpo, on=["code_muni", "ano"], how="left")
base_final["inc_100k"] = (base_final["casos"] / base_final["populacao"]) * 100_000

print(f"-> Base de dados 100% consolidada e pronta para a Inteligência Artificial!")

4/7 - Carregando dados de esgoto e saneamento (SNIS)...
-> Base de dados 100% consolidada e pronta para a Inteligência Artificial!


In [12]:
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

print("\n5/7 - Iniciando Validação e Agrupamento Avançado (K-Means)...")

# 1. Consolidação das médias do período por município
analise_muni = (
    base_final.groupby(["code_muni", "municipality", "state"])
    .agg(
        inc_media_100k=("inc_100k", "mean"),
        casos_totais=("casos", "sum"),
        pop_media=("populacao", "mean"),
        natural_ha=("natural_ha", "mean"),
        hydro_ha=("hydro_ha", "mean"),
        açudes_canais_ha=("açudes_canais_ha", "mean"),
        taxa_esgoto_media=("taxa_esgoto", "mean")
    )
    .reset_index()
)

# 2. Filtro de municípios ativos com dados sanitários válidos
df_ml_brasil = analise_muni[
    (analise_muni["casos_totais"] > 0) & 
    (analise_muni["taxa_esgoto_media"].notna())
].copy()

print(f"-> Total de municípios selecionados para o agrupamento: {len(df_ml_brasil)}")

# 3. Padronização dos dados para balancear as escalas (hectares, habitantes e percentuais)
features_cluster = ["inc_media_100k", "açudes_canais_ha", "taxa_esgoto_media", "pop_media"]
scaler = StandardScaler()
dados_norm = scaler.fit_transform(df_ml_brasil[features_cluster])

# ==============================================================================
# SALVANDO OS DADOS PARA A VALIDAÇÃO TEMPORAL
# ==============================================================================
# Salvamos o dataframe df_ml_brasil que contém todas as médias limpas de 2015-2019 e 2020-2025
"""df_ml_brasil.to_csv("dados_tratados_2015_2019.csv", index=False, encoding="utf-8-sig")

print("\n=======================================================")
print("  SUCESSO: DADOS DE 2015-2019 PRONTOS E EXPORTADOS!    ")
print("=======================================================")
print("O arquivo 'dados_tratados_2015_2019.csv' foi gerado na sua pasta.")
print("Agora você já pode fechar este notebook e voltar para o original (2011-2014).")"""

df_ml_brasil.to_csv("dados_tratados_2020_2025.csv", index=False, encoding="utf-8-sig")

print("\n=======================================================")
print("  SUCESSO: DADOS DE 2020-2025 PRONTOS E EXPORTADOS!    ")
print("=======================================================")
print("O arquivo 'dados_tratados_2020_2025.csv' foi gerado na sua pasta.")
print("Agora você já pode fechar este notebook e voltar para o original (2011-2014).")


5/7 - Iniciando Validação e Agrupamento Avançado (K-Means)...
-> Total de municípios selecionados para o agrupamento: 395

  SUCESSO: DADOS DE 2020-2025 PRONTOS E EXPORTADOS!    
O arquivo 'dados_tratados_2020_2025.csv' foi gerado na sua pasta.
Agora você já pode fechar este notebook e voltar para o original (2011-2014).
